# Clean flight and weather data

Run from the repository root. Download the authorized course data bundle and place it under `data/raw/` using the layout in the README. This notebook writes cleaned Parquet data under `data/clean/`.


In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/project_clean.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the cloned project directory.")
DATA_ROOT = PROJECT_ROOT / "data"

spark = (
    SparkSession.builder
    .appName("flight_cleaning")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [ ]:
WBAN_PATH = str(DATA_ROOT / "raw" / "wban_airport_timezone.csv")

wban_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(WBAN_PATH)
)

wban_lookup = (
    wban_raw
    .select(
        F.trim(F.col("AirportID")).alias("AirportID"),
        F.lpad(F.trim(F.col("WBAN")), 5, "0").alias("WBAN"),
        F.trim(F.col("TimeZone")).alias("TimeZone")
    )
    .dropna(subset=["AirportID", "WBAN", "TimeZone"])
)

# For weather：WBAN -> AirportID
wban_to_airport_df = wban_lookup.select("WBAN", "AirportID", "TimeZone").dropDuplicates(["WBAN"])

# For flight：AirportID -> TimeZone
airport_tz_df = (
    wban_lookup
    .select(
        F.col("AirportID").alias("airport_id"),
        F.col("TimeZone").cast("int").alias("timezone_offset")
    )
    .dropDuplicates(["airport_id"])
)

## Flight

In [ ]:
FLIGHT_RAW_PATH = str(DATA_ROOT / "raw" / "Flights")
FLIGHT_CLEAN_PATH = str(DATA_ROOT / "clean" / "Flights")

In [ ]:
flights_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{FLIGHT_RAW_PATH}/*")
)

print("Raw row count:", flights_raw.count())
print("Raw columns:", flights_raw.columns)
flights_raw.printSchema()
flights_raw.show(5, truncate=False)

In [ ]:
# drop useless columns
drop_cols = ["_c12"]
flights_1 = flights_raw.drop(*drop_cols)

# cast columns to correct types
flights_2 = (
    flights_1
    .withColumn("FL_DATE", F.to_date("FL_DATE", "yyyy-MM-dd"))
    .withColumn("OP_CARRIER_AIRLINE_ID", F.col("OP_CARRIER_AIRLINE_ID").cast("string"))
    .withColumn("OP_CARRIER_FL_NUM", F.col("OP_CARRIER_FL_NUM").cast("string"))
    .withColumn("ORIGIN_AIRPORT_ID", F.col("ORIGIN_AIRPORT_ID").cast("string"))
    .withColumn("DEST_AIRPORT_ID", F.col("DEST_AIRPORT_ID").cast("string"))
    .withColumn("CRS_DEP_TIME", F.col("CRS_DEP_TIME").cast("string"))
    .withColumn("ARR_DELAY_NEW", F.col("ARR_DELAY_NEW").cast("double"))
    .withColumn("CANCELLED", F.col("CANCELLED").cast("double"))
    .withColumn("DIVERTED", F.col("DIVERTED").cast("double"))
    .withColumn("CRS_ELAPSED_TIME", F.col("CRS_ELAPSED_TIME").cast("double"))
    .withColumn("WEATHER_DELAY", F.col("WEATHER_DELAY").cast("double"))
    .withColumn("NAS_DELAY", F.col("NAS_DELAY").cast("double"))
)

flights_2.printSchema()
# flights_2.show(5, truncate=False)

### Missing values

In [ ]:
cols = flights_2.columns

string_cols = [
    f.name for f in flights_2.schema.fields
    if f.name in cols and isinstance(f.dataType, T.StringType)
]

non_string_cols = [c for c in cols if c not in string_cols]

null_check = flights_2.select(
    *[
        F.sum(
            F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)
        ).alias(c)
        for c in string_cols
    ],
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in non_string_cols
    ]
)

null_check.show(truncate=False)

In [ ]:
flights_3 = (
    flights_2
    .filter(F.col("CANCELLED") == 0.0)
    .filter(F.col("DIVERTED") == 0.0)
    .filter(F.col("CRS_DEP_TIME").isNotNull())
    .filter(F.col("CRS_ELAPSED_TIME").isNotNull())
)

# print("Rows after filtering cancelled/diverted/null keys:", flights_3.count())
# flights_3.show(5, truncate=False)

### Handling `CRS_DEP_TIME`

In [ ]:
# Extract hour and minute from CRS_DEP_TIME
flights_4 = (
    flights_3
    .withColumn(
        "CRS_DEP_TIME_4",
        F.lpad(F.regexp_replace(F.col("CRS_DEP_TIME"), r"[^0-9]", ""), 4, "0")
    )
    .withColumn("dep_hour", F.substring("CRS_DEP_TIME_4", 1, 2).cast("int"))
    .withColumn("dep_minute", F.substring("CRS_DEP_TIME_4", 3, 2).cast("int"))
)

# Filter out rows with invalid times (e.g., dep_hour not in 0-23 or dep_minute not in 0-59)
flights_4 = (
    flights_4
    .filter(F.col("dep_hour").between(0, 23))
    .filter(F.col("dep_minute").between(0, 59))
)

# flights_4.select(
#     "CRS_DEP_TIME", "CRS_DEP_TIME_4", "dep_hour", "dep_minute"
# ).show(10, truncate=False)

In [ ]:
# Combine FL_DATE with dep_hour and dep_minute to create a scheduled departure timestamp
flights_5 = (
    flights_4
    .withColumn(
        "scheduled_dep_str",
        F.concat_ws(
            " ",
            F.date_format("FL_DATE", "yyyy-MM-dd"),
            F.concat_ws(
                ":",
                F.lpad(F.col("dep_hour").cast("string"), 2, "0"),
                F.lpad(F.col("dep_minute").cast("string"), 2, "0"),
                F.lit("00")
            )
        )
    )
    .withColumn(
        "scheduled_dep_ts",
        F.to_timestamp("scheduled_dep_str", "yyyy-MM-dd HH:mm:ss")
    )
)

flights_5.select(
    "FL_DATE", "CRS_DEP_TIME", "CRS_DEP_TIME_4", "scheduled_dep_ts"
).show(10, truncate=False)

### Construct arriving time (considering timezone!)

In [ ]:
# join origin timezone
origin_tz_df = airport_tz_df.select(
    F.col("airport_id").alias("ORIGIN_AIRPORT_ID"),
    F.col("timezone_offset").alias("origin_tz")
)

# join destination timezone
dest_tz_df = airport_tz_df.select(
    F.col("airport_id").alias("DEST_AIRPORT_ID"),
    F.col("timezone_offset").alias("dest_tz")
)

In [ ]:
flights_6 = (
    flights_5
    .join(origin_tz_df, on="ORIGIN_AIRPORT_ID", how="left")
    .join(dest_tz_df, on="DEST_AIRPORT_ID", how="left")
)

# first compute arrival timestamp on the departure-local time axis
flights_6 = (
    flights_6
    .withColumn(
        "scheduled_arr_ts_origin_clock",
        F.from_unixtime(
            F.unix_timestamp("scheduled_dep_ts") + F.col("CRS_ELAPSED_TIME") * 60
        ).cast("timestamp")
    )
)

# then adjust to destination local time by timezone difference
flights_6 = (
    flights_6
    .withColumn(
        "tz_diff_hours",
        F.col("dest_tz") - F.col("origin_tz")
    )
    .withColumn(
        "scheduled_arr_ts_dest_local",
        F.from_unixtime(
            F.unix_timestamp("scheduled_arr_ts_origin_clock") + F.col("tz_diff_hours") * 3600
        ).cast("timestamp")
    )
)

flights_6.select(
    "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "scheduled_dep_ts",
    "CRS_ELAPSED_TIME",
    "origin_tz", "dest_tz", "tz_diff_hours",
    "scheduled_arr_ts_origin_clock",
    "scheduled_arr_ts_dest_local"
).show(10, truncate=False)

In [ ]:
# Count how many rows have null tz_diff_hours
# (About 5% of rows have null tz_diff_hours, which means we couldn't find timezone info for either origin or destination airport)
flights_6.select(
    F.sum(F.when(F.col("tz_diff_hours").isNull(), 1).otherwise(0)).alias("null_tz_diff_count")
).show()

In [ ]:
# Drop the lines that have null tz_diff_hours, since we won't be able to accurately determine their arrival times without timezone info
flights_6 = flights_6.filter(F.col("tz_diff_hours").isNotNull())

### Add some new columns (inlucding binary flags)

In [ ]:
flights_7 = (
    flights_6
    .withColumn("flight_year", F.year("FL_DATE"))
    .withColumn("flight_month", F.month("FL_DATE"))
    .withColumn("flight_yyyymm", F.date_format("FL_DATE", "yyyyMM"))
    .withColumn("arr_delay_min", F.coalesce(F.col("ARR_DELAY_NEW"), F.lit(0.0)))
    .withColumn("is_delayed_15", F.when(F.col("arr_delay_min") >= 15, 1).otherwise(0))
    .withColumn("is_delayed_60", F.when(F.col("arr_delay_min") >= 60, 1).otherwise(0))
    .withColumn("has_weather_delay", F.when(F.coalesce(F.col("WEATHER_DELAY"), F.lit(0.0)) > 0, 1).otherwise(0))
    .withColumn("has_nas_delay", F.when(F.coalesce(F.col("NAS_DELAY"), F.lit(0.0)) > 0, 1).otherwise(0))
)

# generate primary key for identifying unique flights
flights_7 = flights_7.withColumn(
    "flight_id",
    F.sha2(F.concat_ws(
        "_",
        F.col("FL_DATE").cast("string"),
        F.col("OP_CARRIER_AIRLINE_ID"),
        F.col("OP_CARRIER_FL_NUM"),
        F.col("ORIGIN_AIRPORT_ID"),
        F.col("DEST_AIRPORT_ID"),
        F.col("CRS_DEP_TIME").cast("string")
    ), 256)
)

### Export to parquet

In [ ]:
final_cols = [
    "flight_id",
    "FL_DATE",
    "flight_year",
    "flight_month",
    "flight_yyyymm",
    "OP_CARRIER_AIRLINE_ID",
    "OP_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "CRS_DEP_TIME",
    "dep_hour",
    "dep_minute",
    "CRS_ELAPSED_TIME",
    "scheduled_dep_ts",
    "origin_tz",
    "dest_tz",
    "tz_diff_hours",
    "scheduled_arr_ts_origin_clock",
    "scheduled_arr_ts_dest_local",
    "ARR_DELAY_NEW",
    "arr_delay_min",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "has_weather_delay",
    "has_nas_delay",
    "is_delayed_15",
    "is_delayed_60"
]

flights_clean = flights_7.select(*final_cols)

flights_clean.printSchema()
# flights_clean.show(10, truncate=False)

In [ ]:
# export cleaned data as Parquet files partitioned by year and month
(
    flights_clean
    .write
    .mode("overwrite")
    .partitionBy("flight_year", "flight_month")
    .parquet(FLIGHT_CLEAN_PATH)
)

In [ ]:
# Check the output files
df_test = spark.read.parquet(
    f"{FLIGHT_CLEAN_PATH}/flight_year=2012/flight_month=1"
)

print(df_test.count())
df_test.show(5, truncate=False)

## Weather

In [ ]:
WEATHER_RAW_PATH = str(DATA_ROOT / "raw" / "Weather")
WEATHER_CLEAN_PATH = str(DATA_ROOT / "clean" / "Weather")

In [ ]:
weather_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{WEATHER_RAW_PATH}/*")
)

print(len(weather_raw.columns))
print(weather_raw.columns)
weather_raw.printSchema()
weather_raw.show(5, truncate=False)

### Clean the strings

In [ ]:
# trim whitespace from string columns
string_cols = [f.name for f in weather_raw.schema.fields if isinstance(f.dataType, T.StringType)]

weather_1 = weather_raw.select(
    *[F.trim(F.col(c)).alias(c) if c in string_cols else F.col(c) for c in weather_raw.columns]
)

# replace empty strings, "M", and all-whitespace with nulls in string columns
def clean_str(colname):
    return (
        F.when(F.col(colname).isNull(), None)
         .when(F.trim(F.col(colname)) == "", None)
         .when(F.upper(F.trim(F.col(colname))) == "M", None)
         .otherwise(F.trim(F.col(colname)))
    )

weather_2 = weather_1.select(
    *[
        clean_str(c).alias(c) if c in string_cols else F.col(c)
        for c in weather_1.columns
    ]
)

In [ ]:
weather_keep_cols = [
    "WBAN",
    "Date",
    "Time",
    "SkyCondition",
    "Visibility",
    "WeatherType",
    "DryBulbFarenheit",
    "WetBulbFarenheit",
    "DewPointFarenheit",
    "RelativeHumidity",
    "WindSpeed",
    "WindDirection",
    "StationPressure",
    "SeaLevelPressure",
    "HourlyPrecip",
    "Altimeter"
]

weather_3 = weather_2.select(*weather_keep_cols)
weather_3.show(5, truncate=False)

In [ ]:
# Deal with time columns

weather_4 = (
    weather_3
    .withColumn("Date_str", F.col("Date"))
    .withColumn("obs_date", F.to_date("Date", "yyyyMMdd"))
    .withColumn("Time_4", F.lpad(F.col("Time"), 4, "0"))
)

weather_4 = (
    weather_4
    .withColumn("obs_hour", F.substring("Time_4", 1, 2).cast("int"))
    .withColumn("obs_minute", F.substring("Time_4", 3, 2).cast("int"))
)

weather_4 = (
    weather_4
    .filter(F.col("obs_date").isNotNull())
    .filter(F.col("Time_4").isNotNull())
    .filter(F.col("obs_hour").between(0, 23))
    .filter(F.col("obs_minute").between(0, 59))
)

weather_5 = (
    weather_4
    .withColumn(
        "obs_ts_str",
        F.concat_ws(
            " ",
            F.date_format("obs_date", "yyyy-MM-dd"),
            F.concat_ws(
                ":",
                F.lpad(F.col("obs_hour").cast("string"), 2, "0"),
                F.lpad(F.col("obs_minute").cast("string"), 2, "0"),
                F.lit("00")
            )
        )
    )
    .withColumn("obs_ts", F.to_timestamp("obs_ts_str", "yyyy-MM-dd HH:mm:ss"))
)

In [ ]:
# Clean numeric columns by removing non-numeric characters and casting to double
def clean_numeric(colname):
    cleaned = F.trim(F.col(colname))
    cleaned = F.regexp_replace(cleaned, r"[^0-9.\-]", "")

    return (
        F.when(F.col(colname).isNull(), None)
         .when(F.trim(F.col(colname)) == "", None)
         .when(F.upper(F.trim(F.col(colname))) == "M", None)
         .when(cleaned == "", None)
         .otherwise(cleaned.cast("double"))
    )

weather_6 = (
    weather_5
    .withColumn("Visibility", clean_numeric("Visibility"))
    .withColumn("DryBulbFarenheit", clean_numeric("DryBulbFarenheit"))
    .withColumn("WetBulbFarenheit", clean_numeric("WetBulbFarenheit"))
    .withColumn("DewPointFarenheit", clean_numeric("DewPointFarenheit"))
    .withColumn("RelativeHumidity", clean_numeric("RelativeHumidity"))
    .withColumn("WindSpeed", clean_numeric("WindSpeed"))
    .withColumn("WindDirection", clean_numeric("WindDirection"))
    .withColumn("StationPressure", clean_numeric("StationPressure"))
    .withColumn("SeaLevelPressure", clean_numeric("SeaLevelPressure"))
    .withColumn("HourlyPrecip", clean_numeric("HourlyPrecip"))
    .withColumn("Altimeter", clean_numeric("Altimeter"))
)

weather_6.printSchema()
weather_6.show(5, truncate=False)

### Join wban info

In [ ]:
WBAN_PATH = str(DATA_ROOT / "raw" / "wban_airport_timezone.csv")

wban_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(WBAN_PATH)
)

wban_df = (
    wban_df
    .select(
        F.trim(F.col("AirportID")).alias("AirportID"),
        F.lpad(F.trim(F.col("WBAN")), 5, "0").alias("WBAN"),
        F.trim(F.col("TimeZone")).alias("TimeZone")
    )
    .dropna(subset=["AirportID", "WBAN"])
)

In [ ]:
weather_7 = (
    weather_6
    .join(wban_df, on="WBAN", how="inner")
)

### Export to parquet

In [ ]:
weather_8 = (
    weather_7
    .withColumn("weather_year", F.year("obs_date"))
    .withColumn("weather_month", F.month("obs_date"))
    .withColumn("weather_yyyymm", F.date_format("obs_date", "yyyyMM"))
)

In [ ]:
weather_final_cols = [
    "WBAN",
    "AirportID",
    "TimeZone",
    "obs_date",
    "Time_4",
    "obs_hour",
    "obs_minute",
    "obs_ts",
    "SkyCondition",
    "Visibility",
    "WeatherType",
    "DryBulbFarenheit",
    "WetBulbFarenheit",
    "DewPointFarenheit",
    "RelativeHumidity",
    "WindSpeed",
    "WindDirection",
    "StationPressure",
    "SeaLevelPressure",
    "HourlyPrecip",
    "Altimeter",
    "weather_year",
    "weather_month",
    "weather_yyyymm"
]

weather_clean = weather_8.select(*weather_final_cols)
weather_clean.printSchema()
weather_clean.show(5, truncate=False)

In [ ]:
weather_clean.select(
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in [
            "AirportID", "obs_ts", "Visibility", "DryBulbFarenheit",
            "RelativeHumidity", "WindSpeed", "Altimeter"
        ]
    ]
).show(truncate=False)

In [ ]:
(
    weather_clean
    .write
    .mode("overwrite")
    .partitionBy("weather_year", "weather_month")
    .parquet(WEATHER_CLEAN_PATH)
)